In [1]:
print("hello")

hello


In [3]:
import pandas as pd

amazon_conversations = pd.read_csv(
    "../data/processed/amazon_conversations.csv"
)

print("Columns:")
print(amazon_conversations.columns.tolist())

print("\nShape:")
print(amazon_conversations.shape)

print("\nFirst row:")
display(amazon_conversations.head(1))

Columns:
['root_tweet_id', 'thread_size', 'conversation']

Shape:
(76799, 3)

First row:


,root_tweet_id,thread_size,conversation
0,272,6,CUSTOMER: amazonのfireTVstickが見れない😢\n\nAMAZON: ...


In [5]:
import re
import pandas as pd

# Only consider small message-part numbers such as 1/2, 2/2, 1/3, etc.
pattern = r"(?<!\d)([1-9])\s*/\s*([2-9])(?!\d)"

part_rows = []

for _, row in amazon_conversations.iterrows():
    matches = re.findall(pattern, str(row["conversation"]))
    
    for part, total in matches:
        part = int(part)
        total = int(total)
        
        # Keep only logically valid part numbers
        if part <= total:
            part_rows.append({
                "root_tweet_id": row["root_tweet_id"],
                "part": part,
                "total_parts": total,
                "conversation": row["conversation"]
            })

parts_df = pd.DataFrame(part_rows)

print("Total conversations:", len(amazon_conversations))
print(
    "Conversations containing realistic X/Y patterns:",
    parts_df["root_tweet_id"].nunique() if len(parts_df) else 0
)
print("Total realistic X/Y occurrences:", len(parts_df))

if len(parts_df) > 0:
    print("\nPatterns found:")
    
    pattern_counts = (
        parts_df
        .groupby(["part", "total_parts"])
        .size()
        .reset_index(name="count")
        .sort_values(["total_parts", "part"])
    )
    
    display(pattern_counts)

    print("\nExamples:")
    display(
        parts_df[
            ["root_tweet_id", "part", "total_parts", "conversation"]
        ].head(10)
    )
else:
    print("No realistic X/Y patterns found.")

Total conversations: 76799
Conversations containing realistic X/Y patterns: 8551
Total realistic X/Y occurrences: 20709

Patterns found:


,part,total_parts,count
0,1,2,8941
8,2,2,8632
1,1,3,981
9,2,3,979
13,3,3,989
2,1,4,36
10,2,4,22
14,3,4,41
17,4,4,34
3,1,5,6



Examples:


,root_tweet_id,part,total_parts,conversation
0,690,1,2,CUSTOMER: @115850 @115821 @AmazonHelp @115851 ...
1,690,2,2,CUSTOMER: @115850 @115821 @AmazonHelp @115851 ...
2,690,1,2,CUSTOMER: @115850 @115821 @AmazonHelp @115851 ...
3,690,2,2,CUSTOMER: @115850 @115821 @AmazonHelp @115851 ...
4,1748,1,2,CUSTOMER: @AmazonHelp that is not my apartment...
5,1748,2,2,CUSTOMER: @AmazonHelp that is not my apartment...
6,5789,1,2,CUSTOMER: @115850 I have stopped ordering from...
7,5789,2,2,CUSTOMER: @115850 I have stopped ordering from...
8,5796,1,2,CUSTOMER: I go to watch #thebay and episode 7 ...
9,5796,2,2,CUSTOMER: I go to watch #thebay and episode 7 ...


Pattern	Occurrences
1/2	8,941
2/2	8,632
1/3	981
2/3	979
3/3	989

Then there are a small number of 4-part, 5-part, etc. messages.

So this strongly confirms that the dataset contains genuine multi-part Twitter responses.

In [6]:
# Get conversations containing 1/2 or 2/2
mask = amazon_conversations["conversation"].str.contains(
    r"(?<!\d)[12]\s*/\s*2(?!\d)",
    regex=True,
    na=False
)

part_conversations = amazon_conversations[mask]

print("Conversations containing 1/2 or 2/2:",
      len(part_conversations))

print("\n--- 5 REAL EXAMPLES ---\n")

for _, row in part_conversations.sample(
    min(5, len(part_conversations)),
    random_state=42
).iterrows():
    
    print("=" * 80)
    print("ROOT TWEET ID:", row["root_tweet_id"])
    print("THREAD SIZE:", row["thread_size"])
    print("-" * 80)
    print(row["conversation"])
    print()

Conversations containing 1/2 or 2/2: 7729

--- 5 REAL EXAMPLES ---

ROOT TWEET ID: 682459
THREAD SIZE: 11
--------------------------------------------------------------------------------
CUSTOMER: @115850 @AmazonHelp Track my mail on cs-reply fr current orde via ATS, let me see how you'll live up to your day before yesterdays promise

AMAZON: @184612 Our team is working on this. Kindly reply to the correspondence that we've sent. ^SG

CUSTOMER: @AmazonHelp as suspected now d shpmnts gt updated as "I rescheduled" so mch evn aftr giving prior info 2 ur team tht ths is wht is gng 2 happen in d end

AMAZON: @184612 I understand your concern regarding the delivery of your order. I’d like to help you; please fill this form:  ^SU(1/2)

AMAZON: @184612 https://t.co/beaaDm0muc and I’ll contact you soon.^SU(2/2)

CUSTOMER: @AmazonHelp Done 👍
2 Forms within 7 days for your wholly owned logistics ATS certainly speaks something indirectly.

AMAZON: @184612 Thank you for writing back to us. We will 

How do we identify which Amazon tweets belong to the same multi-part response?


RAW/PROCESSED CONVERSATION
          ↓
     Keep original
          ↓
   Retrieval preprocessing
          ↓
Identify multi-part response
          ↓
Combine only when needed
          ↓
Historical evidence



so that we dont loosee info